# Reto proyecto: Practica los foundation models en NLP

**Objetivo del notebook:** usar un foundation model de NLP mediante la API de OpenAI para generar salidas estructuradas en JSON, definir schemas con `pydantic`, validar las respuestas y aplicar el resultado a un caso real de ecommerce: generación de descripciones SEO para productos o servicios.

> Nota de seguridad: el notebook usa una API key ficticia por defecto (`sk-YOUR_FAKE_API_KEY`). Para ejecutar la llamada real a la API, configura tu clave como variable de entorno `OPENAI_API_KEY` y cambia `USE_REAL_API = True`.

## 1. Preparación del entorno

Dependencias principales:

```bash
pip install openai pydantic
```

En Google Colab puedes descomentar la siguiente celda si las librerías no están instaladas.

In [1]:
# En Google Colab o en un entorno nuevo, descomenta esta línea:
# !pip install -q "openai>=1.54.0" "pydantic>=2.7"

## 2. Importación de librerías y configuración

Se define un modo demo para que el notebook pueda ejecutarse completo sin consumir API. El código de llamada real queda implementado en la función `generate_product_seo`.

In [2]:
from __future__ import annotations

import copy
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, Field, ValidationError, field_validator

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

# Configuración general del proyecto
MODEL_NAME = "gpt-4o-2024-08-06"  # Modelo compatible con structured outputs
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "sk-YOUR_FAKE_API_KEY")
USE_REAL_API = False  # Cambiar a True solo si tienes una API key real configurada

OUTPUT_DIR = Path("generated_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Modo API real activado: {USE_REAL_API}")
print(f"Directorio de salida: {OUTPUT_DIR.resolve()}")

Modo API real activado: False
Directorio de salida: /generated_outputs


## 3. Definición de schemas con Pydantic

El proyecto usa cuatro clases relacionadas:

1. `KeywordSet`: palabras clave principales, secundarias y long-tail.
2. `SEODescription`: ficha de descripción SEO del producto.
3. `GenerationMetadata`: metadatos de generación.
4. `ProductSEOResult`: schema principal que agrupa toda la salida.

Los validadores comprueban reglas de calidad: puntuación SEO válida, número mínimo de bullets, longitud razonable de metatítulo y metadescripción, y listas de palabras clave no vacías.

In [3]:
class KeywordSet(BaseModel):
    primary: str = Field(..., description="Palabra clave principal del producto o servicio")
    secondary: list[str] = Field(..., description="Palabras clave secundarias")
    long_tail: list[str] = Field(..., description="Consultas long-tail con intención de búsqueda")
    negative_keywords: list[str] = Field(..., description="Términos que se deben evitar por baja relevancia o intención incorrecta")

    @field_validator("primary")
    @classmethod
    def primary_not_empty(cls, value: str) -> str:
        if not value.strip():
            raise ValueError("La palabra clave principal no puede estar vacía.")
        return value.strip()

    @field_validator("secondary", "long_tail", "negative_keywords")
    @classmethod
    def keyword_lists_not_empty(cls, value: list[str]) -> list[str]:
        if len(value) == 0:
            raise ValueError("Las listas de keywords deben contener al menos un elemento.")
        cleaned = [item.strip() for item in value]
        if any(not item for item in cleaned):
            raise ValueError("Las keywords no pueden contener cadenas vacías.")
        return cleaned


class SEODescription(BaseModel):
    meta_title: str = Field(..., description="Título SEO para buscadores")
    meta_description: str = Field(..., description="Metadescripción SEO")
    short_description: str = Field(..., description="Descripción comercial breve")
    bullet_points: list[str] = Field(..., description="Puntos destacados del producto")
    call_to_action: str = Field(..., description="Llamada a la acción final")
    seo_score: int = Field(..., description="Puntuación SEO estimada de 0 a 100")
    tone: Literal["profesional", "cercano", "premium", "técnico"] = Field(..., description="Tono comunicativo")
    search_intent: Literal["informacional", "comercial", "transaccional", "navegacional"] = Field(..., description="Intención de búsqueda objetivo")

    @field_validator("seo_score")
    @classmethod
    def score_in_range(cls, value: int) -> int:
        if not 0 <= value <= 100:
            raise ValueError("seo_score debe estar entre 0 y 100.")
        return value

    @field_validator("bullet_points")
    @classmethod
    def enough_bullets(cls, value: list[str]) -> list[str]:
        if len(value) < 3:
            raise ValueError("Debe haber al menos 3 bullet points.")
        return [item.strip() for item in value]

    @field_validator("meta_title")
    @classmethod
    def meta_title_reasonable_length(cls, value: str) -> str:
        if len(value) > 70:
            raise ValueError("El meta_title no debe superar 70 caracteres.")
        return value.strip()

    @field_validator("meta_description")
    @classmethod
    def meta_description_reasonable_length(cls, value: str) -> str:
        if len(value) > 170:
            raise ValueError("La meta_description no debe superar 170 caracteres.")
        return value.strip()


class GenerationMetadata(BaseModel):
    generated_at: str = Field(..., description="Fecha de generación en formato ISO")
    model: str = Field(..., description="Modelo usado para generar la respuesta")
    language: Literal["es", "en"] = Field(..., description="Idioma de la respuesta")
    industry: str = Field(..., description="Industria o sector del caso de uso")
    api_mode: Literal["real_api", "demo_mock"] = Field(..., description="Indica si se usó API real o respuesta simulada")
    request_id: str = Field(..., description="Identificador interno de la petición")


class ProductSEOResult(BaseModel):
    product_name: str = Field(..., description="Nombre del producto o servicio")
    category: str = Field(..., description="Categoría comercial")
    target_customer: str = Field(..., description="Cliente objetivo")
    keywords: KeywordSet = Field(..., description="Conjunto de palabras clave SEO")
    description: SEODescription = Field(..., description="Contenido SEO generado")
    metadata: GenerationMetadata = Field(..., description="Metadatos de generación")
    validation_notes: list[str] = Field(..., description="Notas de validación y utilidad del resultado")

    @field_validator("product_name", "category", "target_customer")
    @classmethod
    def text_fields_not_empty(cls, value: str) -> str:
        if not value.strip():
            raise ValueError("Los campos de texto principales no pueden estar vacíos.")
        return value.strip()

    @field_validator("validation_notes")
    @classmethod
    def notes_not_empty(cls, value: list[str]) -> list[str]:
        if len(value) == 0:
            raise ValueError("validation_notes debe incluir al menos una nota.")
        return [item.strip() for item in value]

print("Schemas definidos correctamente.")
print(ProductSEOResult.model_json_schema()["title"])

Schemas definidos correctamente.
ProductSEOResult


## 4. Funciones auxiliares

Estas funciones construyen el prompt, validan la respuesta y generan una salida demo cuando no se quiere llamar a la API real.

In [4]:
def to_dict(model: BaseModel) -> dict:
    """Convierte un modelo Pydantic en diccionario compatible con Pydantic v2."""
    return model.model_dump()


def truncate(text: str, max_len: int) -> str:
    """Recorta texto sin romper demasiado la legibilidad."""
    text = " ".join(text.split())
    if len(text) <= max_len:
        return text
    return text[: max_len - 1].rstrip() + "…"


def slugify(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-záéíóúñ0-9]+", "-", text)
    return text.strip("-")


def build_user_prompt(product_name: str, category: str, target_customer: str, differentiators: list[str]) -> str:
    differentiators_txt = ", ".join(differentiators)
    return f"""
Genera una ficha SEO estructurada para ecommerce en español.

Producto o servicio: {product_name}
Categoría: {category}
Cliente objetivo: {target_customer}
Diferenciadores: {differentiators_txt}

Requisitos:
- Devuelve la información siguiendo exactamente el schema Pydantic solicitado.
- Incluye palabras clave relevantes y realistas.
- La descripción debe estar optimizada para SEO, pero sonar natural.
- Evita claims médicos, legales o imposibles de verificar.
- Usa un tono comercial y profesional.
- El meta_title debe ser menor o igual a 70 caracteres.
- La meta_description debe ser menor o igual a 170 caracteres.
""".strip()


def validate_product_result(data: dict) -> ProductSEOResult | None:
    """Valida un diccionario contra el schema principal y muestra errores de forma clara."""
    try:
        validated = ProductSEOResult.model_validate(data)
        print("✅ Validación correcta: el JSON cumple el schema ProductSEOResult.")
        return validated
    except ValidationError as error:
        print("❌ Validación fallida. Errores encontrados:")
        for issue in error.errors():
            loc = " -> ".join(str(part) for part in issue.get("loc", []))
            print(f"- Campo: {loc} | Error: {issue.get('msg')}")
        return None


def generate_mock_result(product_name: str, category: str, target_customer: str, differentiators: list[str]) -> ProductSEOResult:
    """Respuesta simulada para poder ejecutar el notebook completo sin API key real."""
    primary = f"{product_name.lower()} {category.lower()}"
    differentiator = differentiators[0] if differentiators else "diseño funcional"
    title = truncate(f"{product_name}: {category} premium", 70)
    meta = truncate(
        f"Descubre {product_name}, una solución de {category} para {target_customer}, con {differentiator} y compra fácil online.",
        170,
    )
    result = ProductSEOResult(
        product_name=product_name,
        category=category,
        target_customer=target_customer,
        keywords=KeywordSet(
            primary=primary,
            secondary=[
                f"comprar {product_name.lower()}",
                f"{category.lower()} online",
                f"mejor {category.lower()}",
                f"{product_name.lower()} opiniones",
            ],
            long_tail=[
                f"{product_name.lower()} para {target_customer.lower()}",
                f"dónde comprar {product_name.lower()} online",
                f"{category.lower()} con {differentiator.lower()}",
            ],
            negative_keywords=["gratis", "segunda mano", "manual pirata"],
        ),
        description=SEODescription(
            meta_title=title,
            meta_description=meta,
            short_description=(
                f"{product_name} es una propuesta de {category} pensada para {target_customer}. "
                f"Combina {', '.join(differentiators[:3])} con una presentación clara para facilitar la compra online."
            ),
            bullet_points=[
                f"Diseñado para {target_customer}.",
                f"Diferenciador principal: {differentiator}.",
                "Descripción preparada para mejorar la visibilidad en buscadores.",
                "Estructura clara para ficha de producto, anuncios y landing pages.",
            ],
            call_to_action=f"Añade {product_name} a tu catálogo y mejora la conversión de tu ecommerce.",
            seo_score=88,
            tone="profesional",
            search_intent="transaccional",
        ),
        metadata=GenerationMetadata(
            generated_at=datetime.now(timezone.utc).isoformat(),
            model=MODEL_NAME,
            language="es",
            industry=category,
            api_mode="demo_mock",
            request_id=f"demo-{slugify(product_name)}",
        ),
        validation_notes=[
            "La salida cumple el schema esperado.",
            "El contenido está orientado a ecommerce y SEO.",
            "La respuesta demo permite ejecutar el notebook sin API key real.",
        ],
    )
    return result

print("Funciones auxiliares cargadas.")

Funciones auxiliares cargadas.


## 5. Implementación de la llamada a la API de OpenAI

La función `generate_product_seo` implementa la llamada real con `client.responses.parse(...)` y `text_format=ProductSEOResult`. Esto permite que la API devuelva una respuesta ya estructurada según el schema de Pydantic.

Por defecto, el notebook usa `USE_REAL_API = False` para evitar exponer claves y para que pueda ejecutarse sin coste. Para probarlo con la API real:

1. Configura `OPENAI_API_KEY` como variable de entorno.
2. Cambia `USE_REAL_API = True` en la celda de configuración.
3. Ejecuta de nuevo el notebook.

In [5]:
def generate_product_seo(
    product_name: str,
    category: str,
    target_customer: str,
    differentiators: list[str],
    use_real_api: bool = USE_REAL_API,
) -> ProductSEOResult:
    """Genera una ficha SEO estructurada usando OpenAI Structured Outputs o una demo local.

    Si use_real_api=True, se llama a la API de OpenAI usando un modelo compatible
    con structured outputs. Si use_real_api=False, se genera una respuesta demo válida.
    """
    if not use_real_api:
        return generate_mock_result(product_name, category, target_customer, differentiators)

    if OpenAI is None:
        raise ImportError("La librería openai no está instalada. Ejecuta: pip install openai")

    if OPENAI_API_KEY == "sk-YOUR_FAKE_API_KEY":
        raise ValueError(
            "Configura una API key real en la variable de entorno OPENAI_API_KEY antes de usar la API."
        )

    client = OpenAI(api_key=OPENAI_API_KEY)

    system_prompt = (
        "Eres un especialista senior en SEO para ecommerce. "
        "Generas fichas de producto útiles, verificables, comerciales y estructuradas. "
        "No inventes certificaciones, garantías ni datos técnicos no proporcionados."
    )

    response = client.responses.parse(
        model=MODEL_NAME,
        input=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": build_user_prompt(product_name, category, target_customer, differentiators),
            },
        ],
        text_format=ProductSEOResult,
    )

    parsed_result = response.output_parsed

    # Aunque la API ya respeta el schema, validamos explícitamente para cumplir el requisito del proyecto.
    validated_result = ProductSEOResult.model_validate(to_dict(parsed_result))
    return validated_result

print("Función de API implementada correctamente.")

Función de API implementada correctamente.


## 6. Caso de uso práctico: generación SEO para ecommerce

Se prueban dos productos/servicios ficticios:

1. Un reloj inteligente orientado a salud y deporte.
2. Un servicio de limpieza ecológica a domicilio.

El objetivo es demostrar que el mismo foundation model puede adaptarse a industrias distintas manteniendo una salida JSON estable.

In [6]:
test_cases = [
    {
        "product_name": "VitalBand Pro",
        "category": "reloj inteligente deportivo",
        "target_customer": "deportistas amateurs y usuarios que quieren monitorizar su bienestar diario",
        "differentiators": ["batería de 10 días", "monitorización de sueño", "resistencia al agua", "diseño ligero"],
    },
    {
        "product_name": "EcoClean 360",
        "category": "servicio de limpieza ecológica a domicilio",
        "target_customer": "familias y profesionales que buscan limpieza sostenible en grandes ciudades",
        "differentiators": ["productos biodegradables", "reserva online", "personal verificado", "planes semanales flexibles"],
    },
]

results: list[ProductSEOResult] = []

for case in test_cases:
    result = generate_product_seo(**case)
    results.append(result)
    output_path = OUTPUT_DIR / f"{slugify(result.product_name)}_seo.json"
    output_path.write_text(json.dumps(to_dict(result), ensure_ascii=False, indent=2), encoding="utf-8")
    print("=" * 80)
    print(f"Producto: {result.product_name}")
    print(json.dumps(to_dict(result), ensure_ascii=False, indent=2))
    print(f"Guardado en: {output_path}")

Producto: VitalBand Pro
{
  "product_name": "VitalBand Pro",
  "category": "reloj inteligente deportivo",
  "target_customer": "deportistas amateurs y usuarios que quieren monitorizar su bienestar diario",
  "keywords": {
    "primary": "vitalband pro reloj inteligente deportivo",
    "secondary": [
      "comprar vitalband pro",
      "reloj inteligente deportivo online",
      "mejor reloj inteligente deportivo",
      "vitalband pro opiniones"
    ],
    "long_tail": [
      "vitalband pro para deportistas amateurs y usuarios que quieren monitorizar su bienestar diario",
      "dónde comprar vitalband pro online",
      "reloj inteligente deportivo con batería de 10 días"
    ],
    "negative_keywords": [
      "gratis",
      "segunda mano",
      "manual pirata"
    ]
  },
  "description": {
    "meta_title": "VitalBand Pro: reloj inteligente deportivo premium",
    "meta_description": "Descubre VitalBand Pro, una solución de reloj inteligente deportivo para deportistas amateurs y

## 7. Validación explícita del JSON devuelto

Esta sección demuestra dos escenarios:

- Validación correcta de una respuesta generada.
- Validación fallida cuando el JSON no respeta el schema.

In [7]:
print("Validación de una respuesta correcta:")
valid_json = to_dict(results[0])
validated = validate_product_result(valid_json)

print("\nPrueba con JSON inválido: seo_score fuera de rango y pocos bullet points")
invalid_json = copy.deepcopy(valid_json)
invalid_json["description"]["seo_score"] = 130
invalid_json["description"]["bullet_points"] = ["Solo un punto"]
_ = validate_product_result(invalid_json)

Validación de una respuesta correcta:
✅ Validación correcta: el JSON cumple el schema ProductSEOResult.

Prueba con JSON inválido: seo_score fuera de rango y pocos bullet points
❌ Validación fallida. Errores encontrados:
- Campo: description -> bullet_points | Error: Value error, Debe haber al menos 3 bullet points.
- Campo: description -> seo_score | Error: Value error, seo_score debe estar entre 0 y 100.


## 8. Pruebas automáticas básicas

Estas comprobaciones ayudan a asegurar que el notebook produce resultados válidos y utilizables.

In [8]:
assert len(results) == 2, "Deben generarse dos resultados de prueba."

for result in results:
    assert isinstance(result, ProductSEOResult)
    assert result.description.seo_score >= 0 and result.description.seo_score <= 100
    assert len(result.description.bullet_points) >= 3
    assert result.keywords.primary
    assert len(result.keywords.secondary) >= 1
    assert len(result.keywords.long_tail) >= 1
    assert result.metadata.language == "es"

print("✅ Todas las pruebas básicas se han superado.")

✅ Todas las pruebas básicas se han superado.


## 9. Análisis de aplicación en un entorno real

Este sistema sería útil en una plataforma de ecommerce porque permite automatizar fichas de producto con una estructura constante. En vez de recibir texto libre difícil de integrar en una base de datos, la API devuelve un JSON validado con campos específicos: título SEO, metadescripción, palabras clave, puntos destacados, llamada a la acción y metadatos de generación.

**Ventajas principales:**

- Reduce el tiempo de creación de fichas de producto.
- Mantiene una estructura uniforme para todos los artículos.
- Facilita guardar la salida en una base de datos o CMS.
- Permite validar automáticamente si el contenido cumple reglas mínimas de calidad.
- Se puede adaptar a diferentes sectores, como tecnología, servicios locales, moda o alimentación.

**Riesgos y mitigaciones:**

- Riesgo de datos inventados: se reduce indicando al modelo que no cree certificaciones ni características no proporcionadas.
- Riesgo de JSON inválido: se mitiga con Structured Outputs y validación Pydantic.
- Riesgo de baja calidad SEO: se mitiga usando campos obligatorios, puntuación SEO y revisión humana en productos críticos.
- Riesgo de exposición de credenciales: se evita usando variables de entorno y nunca guardando una API key real en el notebook.

## 10. Conclusión

El proyecto cumple los requisitos principales:

- Define schemas claros y relacionados con `pydantic`.
- Implementa una llamada real a la API de OpenAI con structured outputs.
- Valida explícitamente el JSON contra el schema.
- Incluye manejo de errores de validación.
- Aplica el resultado a un caso práctico de ecommerce con descripciones optimizadas para SEO.
- Ejecuta pruebas con dos productos/servicios distintos.